# CETR-Net V2 Stage-2 Topology Ablation
Private GPU research notebook, FIVES V2 train/validation only. This is not a held-out official-test experiment. The current pilot uses one pretrained backbone and multiple refiner seeds; it does not prove complete independent multi-seed architecture training.


In [ ]:
import os,sys,json,csv,zipfile,shutil,time
from pathlib import Path
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
import numpy as np,cv2,tensorflow as tf
import skimage
gpus=tf.config.list_physical_devices("GPU")
print("STAGE2_ENV",{"tf":tf.__version__,"gpu":str(gpus),"skimage":skimage.__version__},flush=True)
assert gpus,"GPU unavailable"
for gpu in gpus:
    try:tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError: pass
INPUT=Path("/kaggle/input/datasets/ardhanabdillah/cetr-fives-v2-private-dev")
if not INPUT.exists():
    for candidate in (Path("/kaggle/input/cetr-fives-v2-private-dev"),):
        if candidate.exists(): INPUT=candidate
if not INPUT.exists():
    matches=[]
    for here,ds,fs in os.walk("/kaggle/input",followlinks=True):
        if len(Path(here).relative_to("/kaggle/input").parts)>7:
            ds.clear()
            continue
        if "fives_v2_trainval_512.zip" in fs:matches.append(Path(here))
    print("FOUND_INPUTS",[str(m) for m in matches])
    if len(matches)==1:INPUT=matches[0]
assert INPUT.exists(),"Private dataset not mounted"
WORK=Path("/kaggle/working/cetr_v2")
WORK.mkdir(parents=True,exist_ok=True)
print("DATASET_PATH",INPUT,flush=True)


In [ ]:
raw=INPUT/"fives_v2_trainval_512"
if raw.is_dir() and (raw/"record_manifest.json").exists():
    print("KAGGLE_DATASET_EXTRACTED_FOLDER",raw,flush=True)
else:
    archives=list(INPUT.rglob("fives_v2_trainval_512.zip"))
    assert len(archives)==1,"No FIVES V2 image package found"
    raw=WORK/"raw"
    raw.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(raw)
manifest=json.loads((raw/"record_manifest.json").read_text())
splits=json.loads((raw/"seed_splits.json").read_text())
mapping={r["id"]:r for r in manifest}
assert len(mapping)==598
for seed in ("42","123","2025"):
    train=splits[seed]["train"]
    val=splits[seed]["validation"]
    assert len(train)==508 and len(val)==90 and not set(train).intersection(val)
    folder=WORK/"SPLITS_NEARDUP_SAFE_V2"/("seed_"+seed)
    folder.mkdir(parents=True,exist_ok=True)
    for split_name,ids in (("train",train),("validation",val)):
        with (folder/("fives_"+split_name+"_split.csv")).open("w",newline="",encoding="utf8") as f:
            wr=csv.DictWriter(f,fieldnames=["id","image","mask","official_split","label_token"])
            wr.writeheader()
            for rid in ids:
                row=mapping[rid]
                img=raw/"images"/(row["image_key"]+".png")
                mask=raw/"masks"/(rid+".png")
                assert img.is_file() and mask.is_file()
                wr.writerow({"id":rid,"image":str(img),"mask":str(mask),
                             "official_split":"official_train","label_token":row["label"]})
print("FIVES_V2_SPLITS_OK",flush=True)


In [ ]:
from pathlib import Path
import os,json,sys,shutil,time
for name in ("cetr_core.py","train_backbone.py"):
    shutil.copy2(INPUT/name,WORK/name)
sys.path.insert(0,str(WORK))
import train_backbone
train_backbone.WORK=WORK
print("INDEPENDENT_BACKBONE_SEEDS_START",flush=True)
for seed in (123,2025):
    print("INDEPENDENT_BACKBONE_NEW_SEED",seed,flush=True)
    sys.argv=["train_backbone.py","--seed",str(seed),"--epochs","45","--size","512","--patience","15"]
    train_backbone.main()
    run=WORK/"runs"/("seed_"+str(seed))/"warmup_V2_FULL"
    best=run/"best_state.json"
    assert best.exists(),seed
    rec=json.loads(best.read_text())
    print("INDEPENDENT_BACKBONE_SEED_COMPLETE",seed,rec,flush=True)


In [ ]:
import json,shutil
summary={}
for seed in (123,2025):
    path=WORK/"runs"/("seed_"+str(seed))/"warmup_V2_FULL"
    best=json.loads((path/"best_state.json").read_text())
    epochs=(path/"epoch_history.jsonl").read_text().splitlines()
    summary[str(seed)]={"best_epoch":best["best_epoch"],"best_val_soft_dice":best["best_dice"],
                        "epochs_logged":len(epochs),"checkpoint_available":(path/"checkpoint_resume"/"checkpoint").exists()}
assert len(summary)==2
(WORK/"CETR_V2_INDEPENDENT_BACKBONES.json").write_text(json.dumps(summary,indent=2))
print("INDEPENDENT_BACKBONE_SUMMARY",summary,flush=True)
